In [1]:
!git clone https://github.com/iconrealestate77/thebeaverschoice.git
%cd thebeaverschoice
!pip install pandas numpy sqlalchemy python-dotenv smolagents -q

fatal: destination path 'thebeaverschoice' already exists and is not an empty directory.
/content/thebeaverschoice


In [2]:
with open(".env", "w") as f:
    f.write("UDACITY_OPENAI_API_KEY=voc-95234576815876652884116a78f129712882.20924617\n")


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("UDACITY_OPENAI_API_KEY"), (
    "UDACITY_OPENAI_API_KEY not found - add it to a .env file before running."
)
print("Key loaded OK")


Key loaded OK


In [4]:
from dotenv import load_dotenv
load_dotenv(override=True)

key = os.environ.get("UDACITY_OPENAI_API_KEY", "")
print(f"Loaded key: {key[:8]}...{key[-4:]} (length {len(key)})")


Loaded key: voc-9523...4617 (length 49)


In [5]:
!pwd
!ls -la


/content/thebeaverschoice
total 288
drwxr-xr-x 3 root root   4096 Sep  9 04:31 .
drwxr-xr-x 1 root root   4096 Sep  9 04:31 ..
-rw-r--r-- 1 root root     73 Sep  9 04:32 .env
drwxr-xr-x 8 root root   4096 Sep  9 04:31 .git
-rw-r--r-- 1 root root  28491 Sep  9 04:31 project_starter.py
-rw-r--r-- 1 root root  30684 Sep  9 04:31 quote_requests.csv
-rw-r--r-- 1 root root   5825 Sep  9 04:31 quote_requests_sample.csv
-rw-r--r-- 1 root root  57510 Sep  9 04:31 quotes.csv
-rw-r--r-- 1 root root   3584 Sep  9 04:31 README.md
-rw-r--r-- 1 root root     84 Sep  9 04:31 requirements.txt
-rw-r--r-- 1 root root  13791 Sep  9 04:31 test_results.xlsx
-rw-r--r-- 1 root root 122456 Sep  9 04:31 Thebeaverschoice.ipynb


In [6]:
%cd project
!cat requirements.txt


[Errno 2] No such file or directory: 'project'
/content/thebeaverschoice
pandas==2.2.3
typing==3.7.4.3
openai==1.76.0
SQLAlchemy==2.0.40
python-dotenv==1.1.0

In [7]:
!pip install -r requirements.txt -q
!pip install smolagents -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.2/661.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00


In [8]:
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("UDACITY_OPENAI_API_KEY"), (
    "UDACITY_OPENAI_API_KEY not found - add it to a .env file before running."
)


In [9]:
import project_starter as pb

pb.init_database(pb.db_engine)
print("DB initialized OK")

report = pb.generate_financial_report("2025-04-01")
print(report)


DB initialized OK
{'as_of_date': '2025-04-01', 'cash_balance': 45059.7, 'inventory_value': np.float64(4940.299999999999), 'total_assets': np.float64(50000.0), 'inventory_summary': [{'item_name': 'Paper plates', 'stock': np.float64(748.0), 'unit_price': 0.1, 'value': np.float64(74.8)}, {'item_name': '100 lb cover stock', 'stock': np.float64(636.0), 'unit_price': 0.5, 'value': np.float64(318.0)}, {'item_name': 'Glossy paper', 'stock': np.float64(587.0), 'unit_price': 0.2, 'value': np.float64(117.4)}, {'item_name': 'Rolls of banner paper (36-inch width)', 'stock': np.float64(546.0), 'unit_price': 2.5, 'value': np.float64(1365.0)}, {'item_name': 'Photo paper', 'stock': np.float64(423.0), 'unit_price': 0.25, 'value': np.float64(105.75)}, {'item_name': 'Cardstock', 'stock': np.float64(595.0), 'unit_price': 0.15, 'value': np.float64(89.25)}, {'item_name': 'Colored paper', 'stock': np.float64(788.0), 'unit_price': 0.1, 'value': np.float64(78.80000000000001)}, {'item_name': '80 lb text paper', 

## System Architecture

Beaver's Choice Paper Company's order-fulfillment system is a **multi-agent**
setup: one `CodeAgent` orchestrator that plans and sequences the work, and
three `ToolCallingAgent` workers, each scoped to a narrow responsibility and a
small set of tools. The orchestrator never touches the database directly —
every read or write goes through a worker agent's tools.

```mermaid
flowchart TB
    C["Customer request<br/>+ request date"] --> O

    subgraph O["orchestrator (CodeAgent)"]
        direction TB
        O1["1. Ask inventory_agent to check stock for every item"]
        O2["2. Ask quoting_agent to price every item found=True"]
        O3["3. Reject items that are out of catalog, understocked\npast the need-by date, or unaffordable"]
        O4["4. Ask sales_agent to record stock orders / sales"]
        O5["5. Compose the customer-facing response"]
        O1 --> O2 --> O3 --> O4 --> O5
    end

    O -->|"task string"| IA
    O -->|"task string"| QA
    O -->|"task string"| SA

    subgraph IA["inventory_agent (ToolCallingAgent)"]
        IT1["check_inventory\nPurpose: match item + check stock\nWraps: get_stock_level"]
        IT2["check_delivery_date\nPurpose: estimate supplier arrival\nWraps: get_supplier_delivery_date"]
        IT3["get_all_inventory_snapshot\nPurpose: full stock snapshot\nWraps: get_all_inventory"]
    end

    subgraph QA["quoting_agent (ToolCallingAgent)"]
        QT1["search_past_quotes\nPurpose: historical quote lookup\nWraps: search_quote_history"]
        QT2["calculate_quote\nPurpose: compute discount price\nPure calculation: no starter helper"]
    end

    subgraph SA["sales_agent (ToolCallingAgent)"]
        ST1["check_finances\nPurpose: cash + inventory valuation\nWraps: get_cash_balance, get_stock_level"]
        ST2["record_sale\nPurpose: record sale transaction\nWraps: create_transaction"]
        ST3["record_stock_order\nPurpose: record supplier purchase\nWraps: create_transaction"]
        ST4["get_full_financial_report\nPurpose: business report\nWraps: generate_financial_report"]
    end

    IA -->|"stock + found flag"| O
    QA -->|"price quote"| O
    SA -->|"transaction id / balances"| O

    IT1 & IT2 & IT3 -.-> DB[(SQLite via SQLAlchemy)]
    QT1 -.-> DB
    ST1 & ST2 & ST3 & ST4 -.-> DB
```

### Agents

| Agent | Type | Responsibility |
|---|---|---|
| `orchestrator` | `CodeAgent` | Owns the end-to-end order workflow: decides what to check, what to reject, and what to finalize. Holds no tools of its own — it only calls the three managed agents. |
| `inventory_agent` | `ToolCallingAgent` | Answers "do we have it, and if not, when could we get it?" for every requested item. |
| `quoting_agent` | `ToolCallingAgent` | Prices confirmed, in-stock items using bulk discounts and historical pricing for consistency. |
| `sales_agent` | `ToolCallingAgent` | The only agent allowed to write to the database — records sales/reorders and reports cash & inventory value. |

### Tools and their starter helper(s)

| Tool | Used by | Purpose | Wraps starter helper(s) |
|---|---|---|---|
| `check_inventory` | inventory_agent | Fuzzy-matches a free-text item name to the catalog and returns its stock + price | `get_stock_level`, `paper_supplies` |
| `check_delivery_date` | inventory_agent | Estimates a supplier delivery date for a reorder | `get_supplier_delivery_date` |
| `get_all_inventory_snapshot` | inventory_agent | Returns the full current stock snapshot in one call (all items with positive stock as of a date), instead of one item at a time | `get_all_inventory` |
| `search_past_quotes` | quoting_agent | Looks up similar historical quotes so new prices stay consistent | `search_quote_history` |
| `calculate_quote` | quoting_agent | Applies quantity-based bulk discounts to price a line item | (pure calculation, no helper) |
| `check_finances` | sales_agent | Cash balance + inventory valued across *every* item that has ever transacted (patches a gap in the starter report — see note below) | `get_cash_balance`, `get_stock_level` |
| `record_sale` | sales_agent | Writes a sales transaction, which reduces stock and increases cash | `create_transaction` |
| `record_stock_order` | sales_agent | Writes a stock-order transaction, which increases stock and reduces cash | `create_transaction` |
| `get_full_financial_report` | sales_agent | Full report incl. top-5 best sellers, for business-health checks | `generate_financial_report` |

**Why `check_finances` exists alongside `get_full_financial_report`:** the starter
`generate_financial_report` only values items present in the *original seeded
inventory table* (40% catalog coverage). Once the agents reorder catalog items
that weren't in that initial snapshot, `generate_financial_report` silently
under-reports inventory value for those items. `check_finances` fixes that by
valuing every item that has ever appeared in `transactions`. We keep
`get_full_financial_report` too (wrapping the *unmodified* starter helper)
because it's the only place `top_selling_products` is exposed, which is useful
for a manager-facing summary even though its inventory valuation is narrower.


In [10]:
!pwd
!ls -la

/content/thebeaverschoice
total 408
drwxr-xr-x 4 root root   4096 Sep  9 04:32 .
drwxr-xr-x 1 root root   4096 Sep  9 04:31 ..
-rw-r--r-- 1 root root     73 Sep  9 04:32 .env
drwxr-xr-x 8 root root   4096 Sep  9 04:31 .git
-rw-r--r-- 1 root root 114688 Sep  9 04:32 munder_difflin.db
-rw-r--r-- 1 root root  28491 Sep  9 04:31 project_starter.py
drwxr-xr-x 2 root root   4096 Sep  9 04:32 __pycache__
-rw-r--r-- 1 root root  30684 Sep  9 04:31 quote_requests.csv
-rw-r--r-- 1 root root   5825 Sep  9 04:31 quote_requests_sample.csv
-rw-r--r-- 1 root root  57510 Sep  9 04:31 quotes.csv
-rw-r--r-- 1 root root   3584 Sep  9 04:31 README.md
-rw-r--r-- 1 root root     84 Sep  9 04:31 requirements.txt
-rw-r--r-- 1 root root  13791 Sep  9 04:31 test_results.xlsx
-rw-r--r-- 1 root root 122456 Sep  9 04:31 Thebeaverschoice.ipynb


In [11]:
%cd project
!cat requirements.txt

[Errno 2] No such file or directory: 'project'
/content/thebeaverschoice
pandas==2.2.3
typing==3.7.4.3
openai==1.76.0
SQLAlchemy==2.0.40
python-dotenv==1.1.0

In [12]:
!pip install -r requirements.txt -q
!pip install smolagents -q

In [13]:
from dotenv import load_dotenv

# Re-load in case this cell runs in a fresh kernel/session after the %cd above.
load_dotenv()
assert os.environ.get("UDACITY_OPENAI_API_KEY"), (
    "UDACITY_OPENAI_API_KEY not found - add it to a .env file before running."
)

In [14]:
import project_starter as pb

pb.init_database(pb.db_engine)
print("DB initialized OK")

report = pb.generate_financial_report("2025-04-01")
print(report)

DB initialized OK
{'as_of_date': '2025-04-01', 'cash_balance': 45059.7, 'inventory_value': np.float64(4940.299999999999), 'total_assets': np.float64(50000.0), 'inventory_summary': [{'item_name': 'Paper plates', 'stock': np.float64(748.0), 'unit_price': 0.1, 'value': np.float64(74.8)}, {'item_name': '100 lb cover stock', 'stock': np.float64(636.0), 'unit_price': 0.5, 'value': np.float64(318.0)}, {'item_name': 'Glossy paper', 'stock': np.float64(587.0), 'unit_price': 0.2, 'value': np.float64(117.4)}, {'item_name': 'Rolls of banner paper (36-inch width)', 'stock': np.float64(546.0), 'unit_price': 2.5, 'value': np.float64(1365.0)}, {'item_name': 'Photo paper', 'stock': np.float64(423.0), 'unit_price': 0.25, 'value': np.float64(105.75)}, {'item_name': 'Cardstock', 'stock': np.float64(595.0), 'unit_price': 0.15, 'value': np.float64(89.25)}, {'item_name': 'Colored paper', 'stock': np.float64(788.0), 'unit_price': 0.1, 'value': np.float64(78.80000000000001)}, {'item_name': '80 lb text paper', 

In [15]:
import re

STOPWORDS = {"paper", "sheets", "sheet", "of", "the", "a", "an"}

def _tokenize(name: str) -> set:
    return set(re.findall(r"[a-zA-Z0-9]+", name.lower()))

def _size_token(name: str) -> str | None:
    match = re.search(r"\bA[3-6]\b", name, re.IGNORECASE)
    return match.group(0).upper() if match else None

CATALOG_NAMES = [item["item_name"] for item in pb.paper_supplies]

def match_catalog_item(requested_name: str, min_overlap: float = 0.5) -> str | None:
    """
    Match a free-text item name to the closest item in our paper_supplies catalog
    using descriptive-word overlap rather than character similarity. Explicitly
    rejects size mismatches (e.g. A3 vs A4) so we never silently substitute a
    different product than what was requested.

    Args:
        requested_name: the raw item name as written by the customer
        min_overlap: minimum fraction of the candidate's descriptive words that
                     must appear in the request for it to count as a match

    Returns:
        The matching catalog item name, or None if nothing matches closely enough.
    """
    requested_size = _size_token(requested_name)
    requested_tokens = _tokenize(requested_name) - STOPWORDS

    best_match, best_score = None, 0.0

    for candidate in CATALOG_NAMES:
        candidate_size = _size_token(candidate)

        # Reject if request specifies a size our candidate doesn't have, or vice versa
        if requested_size != candidate_size:
            continue

        candidate_tokens = _tokenize(candidate) - STOPWORDS
        if not candidate_tokens:
            continue

        overlap = requested_tokens.intersection(candidate_tokens)
        score = len(overlap) / len(candidate_tokens)

        if score > best_score:
            best_score, best_match = score, candidate

    return best_match if best_score >= min_overlap else None

In [16]:
print(match_catalog_item("heavy cardstock"))
print(match_catalog_item("A3 paper"))
print(match_catalog_item("glossy A4 paper"))
print(match_catalog_item("A4 paper"))
print(match_catalog_item("printer paper"))
print(match_catalog_item("standard printer paper"))

Cardstock
None
A4 paper
A4 paper
None
Standard copy paper


In [17]:
from smolagents import tool

@tool
def check_inventory(item_name: str, as_of_date: str) -> dict:
    """
    Check current stock level for a specific paper/product item as of a given date.
    Automatically matches free-text item names to the closest item in our catalog.

    Args:
        item_name: the name of the item to check (can be approximate/free-text)
        as_of_date: ISO date string (YYYY-MM-DD) for the stock snapshot

    Returns:
        A dict with matched_item, current_stock, unit_price, and found (bool).
        If found is False, this item is not carried in our catalog.
    """
    matched = match_catalog_item(item_name)
    if not matched:
        return {"matched_item": None, "current_stock": 0, "unit_price": None, "found": False}

    stock_df = pb.get_stock_level(matched, as_of_date)
    current_stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0

    unit_price = next(
        (p["unit_price"] for p in pb.paper_supplies if p["item_name"] == matched), None
    )

    return {
        "matched_item": matched,
        "current_stock": current_stock,
        "unit_price": unit_price,
        "found": True,
    }


@tool
def check_delivery_date(as_of_date: str, quantity: int) -> str:
    """
    Estimate the delivery date from a supplier for a given order quantity, starting from a date.

    Args:
        as_of_date: ISO date string (YYYY-MM-DD) representing the order date
        quantity: number of units being ordered

    Returns:
        Estimated delivery date as an ISO date string (YYYY-MM-DD).
    """
    return pb.get_supplier_delivery_date(as_of_date, quantity)

In [18]:
print(check_inventory("cardstock", "2025-04-01"))
print(check_inventory("A3 paper", "2025-04-01"))

{'matched_item': 'Cardstock', 'current_stock': 595, 'unit_price': 0.15, 'found': True}
{'matched_item': None, 'current_stock': 0, 'unit_price': None, 'found': False}


In [19]:
@tool
def search_past_quotes(search_terms: list[str], limit: int = 5) -> list[dict]:
    """
    Search historical quotes for similar past requests, useful for pricing consistency.

    Args:
        search_terms: list of keywords to search for (e.g. item names, event type, job type)
        limit: maximum number of past quotes to return

    Returns:
        A list of matching past quotes with original_request, total_amount,
        quote_explanation, job_type, order_size, event_type, and order_date.
    """
    return pb.search_quote_history(search_terms, limit=limit)


@tool
def calculate_quote(item_name: str, quantity: int, unit_price: float) -> dict:
    """
    Calculate a price quote for a given item and quantity, applying bulk discounts.

    Discount tiers (applied to the line total):
        - 1000+ units: 15% off
        - 500-999 units: 10% off
        - 200-499 units: 5% off
        - under 200 units: no discount

    Args:
        item_name: the catalog item name being quoted
        quantity: number of units requested
        unit_price: price per unit from the catalog

    Returns:
        A dict with item_name, quantity, unit_price, subtotal, discount_pct,
        discount_amount, and line_total.
    """
    subtotal = quantity * unit_price

    if quantity >= 1000:
        discount_pct = 0.15
    elif quantity >= 500:
        discount_pct = 0.10
    elif quantity >= 200:
        discount_pct = 0.05
    else:
        discount_pct = 0.0

    discount_amount = subtotal * discount_pct
    line_total = subtotal - discount_amount

    return {
        "item_name": item_name,
        "quantity": quantity,
        "unit_price": unit_price,
        "subtotal": round(subtotal, 2),
        "discount_pct": discount_pct,
        "discount_amount": round(discount_amount, 2),
        "line_total": round(line_total, 2),
    }

In [20]:
print(calculate_quote("Cardstock", 300, 0.15))
print(calculate_quote("A4 paper", 10000, 0.05))
print(search_past_quotes(["ceremony", "cardstock"], limit=3))

{'item_name': 'Cardstock', 'quantity': 300, 'unit_price': 0.15, 'subtotal': 45.0, 'discount_pct': 0.05, 'discount_amount': 2.25, 'line_total': 42.75}
{'item_name': 'A4 paper', 'quantity': 10000, 'unit_price': 0.05, 'subtotal': 500.0, 'discount_pct': 0.15, 'discount_amount': 75.0, 'line_total': 425.0}
[{'original_request': 'I would like to place an order for 500 sheets of high-quality white cardstock and 1000 sheets of colored printer paper for our upcoming ceremony. Please ensure delivery by April 15, 2025. Thank you.', 'total_amount': 160, 'quote_explanation': "Thank you for your order! For 500 sheets of high-quality white cardstock, we typically charge $0.15 each, bringing the subtotal to $75. For the 1000 sheets of colored printer paper at $0.10 each, the subtotal is $100. To help make your order more budget-friendly, I'm happy to apply a bulk discount, rounding the total for both items down to $160, which is a more manageable figure. We will ensure delivery by April 15, 2025.", 'jo

In [21]:
import pandas as pd

@tool
def check_finances(as_of_date: str) -> dict:
    """
    Get the company's cash balance and a full financial report as of a given date.
    Values ALL catalog items that have ever had a transaction, not just the items
    in the original seeded inventory snapshot (pb.generate_financial_report only
    looks at that snapshot, which under-reports inventory value for items ordered
    outside the initial 40% coverage).

    Args:
        as_of_date: ISO date string (YYYY-MM-DD)

    Returns:
        A dict with cash_balance, inventory_value, total_assets, and inventory_summary.
    """
    cash = pb.get_cash_balance(as_of_date)

    # Every item that has ever had a transaction, not just the seeded inventory table
    item_names = pd.read_sql(
        "SELECT DISTINCT item_name FROM transactions WHERE item_name IS NOT NULL",
        pb.db_engine,
    )["item_name"].tolist()

    price_lookup = {p["item_name"]: p["unit_price"] for p in pb.paper_supplies}

    inventory_value = 0.0
    inventory_summary = []
    for item_name in item_names:
        stock_df = pb.get_stock_level(item_name, as_of_date)
        stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0
        if stock <= 0:
            continue
        unit_price = price_lookup.get(item_name)
        if unit_price is None:
            continue  # not a real catalog item, skip valuation
        value = stock * unit_price
        inventory_value += value
        inventory_summary.append({
            "item_name": item_name,
            "stock": stock,
            "unit_price": unit_price,
            "value": round(value, 2),
        })

    return {
        "as_of_date": as_of_date,
        "cash_balance": round(cash, 2),
        "inventory_value": round(inventory_value, 2),
        "total_assets": round(cash + inventory_value, 2),
        "inventory_summary": inventory_summary,
    }


@tool
def record_sale(item_name: str, quantity: int, total_price: float, date: str) -> dict:
    """
    Record a finalized sale transaction in the company's database.

    Args:
        item_name: the catalog item name sold
        quantity: number of units sold
        total_price: total price charged for this line item
        date: ISO date string (YYYY-MM-DD) of the sale

    Returns:
        A dict with transaction_id and status.
    """
    transaction_id = pb.create_transaction(
        item_name=item_name,
        transaction_type="sales",
        quantity=quantity,
        price=total_price,
        date=date,
    )
    return {"transaction_id": transaction_id, "status": "recorded"}


@tool
def record_stock_order(item_name: str, quantity: int, total_price: float, date: str) -> dict:
    """
    Record a stock reorder transaction (purchasing more inventory from a supplier).

    Args:
        item_name: the catalog item name being reordered
        quantity: number of units ordered
        total_price: total cost of the reorder
        date: ISO date string (YYYY-MM-DD) of the order

    Returns:
        A dict with transaction_id and status.
    """
    transaction_id = pb.create_transaction(
        item_name=item_name,
        transaction_type="stock_orders",
        quantity=quantity,
        price=total_price,
        date=date,
    )
    return {"transaction_id": transaction_id, "status": "recorded"}

In [22]:
print(check_finances("2025-04-01"))
result = record_sale("Cardstock", 5, 0.75, "2025-04-01")
print(result)
print(check_finances("2025-04-01"))

{'as_of_date': '2025-04-01', 'cash_balance': 45059.7, 'inventory_value': 4940.3, 'total_assets': 50000.0, 'inventory_summary': [{'item_name': 'Paper plates', 'stock': 748, 'unit_price': 0.1, 'value': 74.8}, {'item_name': '100 lb cover stock', 'stock': 636, 'unit_price': 0.5, 'value': 318.0}, {'item_name': 'Glossy paper', 'stock': 587, 'unit_price': 0.2, 'value': 117.4}, {'item_name': 'Rolls of banner paper (36-inch width)', 'stock': 546, 'unit_price': 2.5, 'value': 1365.0}, {'item_name': 'Photo paper', 'stock': 423, 'unit_price': 0.25, 'value': 105.75}, {'item_name': 'Cardstock', 'stock': 595, 'unit_price': 0.15, 'value': 89.25}, {'item_name': 'Colored paper', 'stock': 788, 'unit_price': 0.1, 'value': 78.8}, {'item_name': '80 lb text paper', 'stock': 249, 'unit_price': 0.4, 'value': 99.6}, {'item_name': 'Large poster paper (24x36 inches)', 'stock': 699, 'unit_price': 1.0, 'value': 699.0}, {'item_name': 'Table covers', 'stock': 736, 'unit_price': 1.5, 'value': 1104.0}, {'item_name': 'Bu

In [23]:
@tool
def get_all_inventory_snapshot(as_of_date: str) -> dict:
    """
    Get a full snapshot of every item currently in stock (positive quantity)
    as of a given date, in a single call. Useful for a broad stock check
    across the whole catalog instead of looking up one item at a time.

    Args:
        as_of_date: ISO date string (YYYY-MM-DD) for the inventory cutoff

    Returns:
        A dict mapping item_name -> current_stock for every item with
        positive stock as of as_of_date.
    """
    return pb.get_all_inventory(as_of_date)


@tool
def get_full_financial_report(as_of_date: str) -> dict:
    """
    Generate the company's full financial report as of a given date, including
    cash balance, inventory valuation, combined assets, an itemized inventory
    breakdown, and the top 5 best-selling products by revenue. This is the
    unmodified starter report (see check_finances for a version that also
    values items outside the original seeded inventory snapshot).

    Args:
        as_of_date: ISO date string (YYYY-MM-DD)

    Returns:
        A dict with as_of_date, cash_balance, inventory_value, total_assets,
        inventory_summary, and top_selling_products.
    """
    return pb.generate_financial_report(as_of_date)


In [24]:
print(get_all_inventory_snapshot("2025-04-01"))
print(get_full_financial_report("2025-04-01")["top_selling_products"])


{'100 lb cover stock': 636.0, '80 lb text paper': 249.0, 'A4 paper': 272.0, 'Banner paper': 793.0, 'Butcher paper': 365.0, 'Cardstock': 590.0, 'Colored paper': 788.0, 'Crepe paper': 234.0, 'Glossy paper': 587.0, 'Invitation cards': 526.0, 'Kraft paper': 493.0, 'Large poster paper (24x36 inches)': 699.0, 'Paper plates': 748.0, 'Patterned paper': 548.0, 'Photo paper': 423.0, 'Presentation folders': 389.0, 'Rolls of banner paper (36-inch width)': 546.0, 'Table covers': 736.0}
[{'item_name': None, 'total_units': nan, 'total_revenue': 50000.0}, {'item_name': 'Cardstock', 'total_units': 5.0, 'total_revenue': 0.75}]


In [25]:
from smolagents import ToolCallingAgent, CodeAgent, OpenAIServerModel

model = OpenAIServerModel(
    model_id="gpt-4o-mini",
    api_base="https://openai.vocareum.com/v1",
    api_key=os.environ["UDACITY_OPENAI_API_KEY"],
)

# verbosity_level=0 on every agent below: at verbose defaults, smolagents prints a
# rich panel per step per agent, which for a 20-request batch run produces output large
# enough that Colab auto-collapses the cell ("Output hidden; open in Colab to view").
# Our own print() statements inside run_test_scenarios() carry the info a reviewer
# needs (context, dates, balances, response) and stay well under that threshold.
inventory_agent = ToolCallingAgent(
    tools=[check_inventory, check_delivery_date, get_all_inventory_snapshot],
    model=model,
    name="inventory_agent",
    max_steps=5,
    verbosity_level=0,
    description=(
        "Checks current stock levels for requested items and estimates supplier "
        "delivery dates for reorders. Use this to find out if we have enough stock "
        "of an item, or when more stock would arrive if we don't. Use "
        "get_all_inventory_snapshot when you need a broad view of everything "
        "currently in stock rather than one item at a time. "
        "IMPORTANT: only check inventory as of the ONE exact date you are given — "
        "never check multiple different dates or date ranges. If you are not "
        "explicitly given a date in your task, do NOT guess one — report back to "
        "your manager that you need the request date."
    ),
)

quoting_agent = ToolCallingAgent(
    tools=[search_past_quotes, calculate_quote],
    model=model,
    name="quoting_agent",
    max_steps=5,
    verbosity_level=0,
    description=(
        "Generates a price quote for a SPECIFIC item and quantity, applying bulk "
        "discounts and referencing historical quote pricing for consistency. "
        "You MUST call this for every item in the order to get its price BEFORE "
        "any sale can be finalized. ONLY quote items that inventory_agent has "
        "confirmed with found=True. Do NOT quote an item just because "
        "search_past_quotes returns historical quotes mentioning a similar name — "
        "those are unrelated past orders, not proof the item is currently sellable."
    ),
)

sales_agent = ToolCallingAgent(
    tools=[check_finances, record_sale, record_stock_order, get_full_financial_report],
    model=model,
    name="sales_agent",
    max_steps=5,
    verbosity_level=0,
    description=(
        "Finalizes sales transactions by recording them in the database (which "
        "changes the company's cash balance), checks company cash balance and "
        "financial health, and records stock reorders. You MUST call this to "
        "actually complete/record an order after inventory and pricing are confirmed — "
        "an order is NOT complete until sales_agent has recorded it. Use "
        "get_full_financial_report if asked for a business-health summary "
        "(e.g. top-selling products), and check_finances for balance checks "
        "during order processing. "
        "IMPORTANT: every check_finances, record_sale, and record_stock_order call "
        "MUST use the EXACT request date you are given in your task — never invent, "
        "assume, or default to any other date (including today's date). If you are "
        "not explicitly given a date in your task, do NOT guess one — report back "
        "to your manager that you need the request date. NEVER record a sale for an "
        "item that was not confirmed as found=True by inventory_agent."
    ),
)

ORCHESTRATOR_INSTRUCTIONS = """
You are the orchestrator for Beaver's Choice Paper Company's order fulfillment system.

CRITICAL: Managed agents (inventory_agent, quoting_agent, sales_agent) take exactly
ONE argument: a single task string. Call them like inventory_agent("task text here") --
NEVER pass a second positional argument (e.g. NEVER call
inventory_agent("task text", {"item": item}) or inventory_agent(task, extra_dict)).
If you need to pass extra context, put it directly inside the task string itself.

CRITICAL: The customer request below includes a specific request date. You MUST use
this EXACT date (and no other date) for every check_inventory, check_delivery_date,
check_finances, and record_sale/record_stock_order call. NEVER invent, assume, or
default to any other date, including today's date or any date from your own training.
Always pass the date explicitly, as a literal string, into EVERY SINGLE task you give
to a managed agent, including follow-up calls within the same order — do not rely on
the managed agent to infer or remember it. NEVER ask a managed agent to check multiple
dates or date ranges — only ever the one exact request date.

For EVERY customer request, you MUST follow this exact process, in order:

1. Call inventory_agent to check stock levels for EVERY item requested, and get
   delivery date estimates for any items that need reordering. Explicitly state
   the request date in the task you give inventory_agent.

2. For EVERY item that IS in our catalog (found=True) and has EITHER enough stock
   OR an acceptable delivery timeline before the customer's needed-by date, call
   quoting_agent to get a price quote for that item and quantity.

   CRITICAL: If inventory_agent reported found=False for an item, that item is NOT
   in our catalog. You MUST NOT call quoting_agent or sales_agent for it under any
   circumstances, even if quoting_agent's search_past_quotes tool returns historical
   quotes that happen to mention similar item names — those are unrelated past orders,
   not proof the item is currently sellable. A found=False item is ALWAYS rejected
   at step 3, with no further action.

3. If ANY of these conditions apply, the order (or that specific line item) CANNOT
   be fulfilled and must be REJECTED with a clear reason:
   - The item is not in our catalog (found=False)
   - Stock is insufficient AND the supplier delivery date is after the customer's
     needed-by date
   - The company's cash balance (check via sales_agent, passing the request date
     explicitly) is insufficient to justify a large reorder

4. For every item that CAN be fulfilled (found=True AND passes the checks above),
   call sales_agent to record the sale transaction (this is what actually finalizes
   the order and updates cash balance). Use the SAME exact request date for this
   transaction. If a reorder is needed to fulfill demand, also record a stock_order
   transaction via sales_agent before recording the sale.

5. Compose a final customer-facing response that:
   - Lists each item, whether it was fulfilled or rejected, and why
   - States the price for each fulfilled item and the total order price
   - States the expected delivery date
   - Does NOT reveal internal details like exact profit margins or raw error messages

You MUST NOT skip steps 2 and 4 for fulfillable items. An order is only complete
once sales_agent has recorded the transaction(s). You MUST NEVER record a sale or
generate a quote for an item that inventory_agent reported as found=False.
"""

orchestrator = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[inventory_agent, quoting_agent, sales_agent],
    name="orchestrator",
    description="Manages the full customer order workflow for Beaver's Choice Paper Company.",
    max_steps=12,  # bumped from 8: extra headroom while the agent settles into the
                   # single-argument managed-agent calling convention above
    verbosity_level=0,
)

In [26]:
def run_orchestrator(customer_request: str, request_date: str) -> str:
    full_task = (
        f"{ORCHESTRATOR_INSTRUCTIONS}\n\n"
        f"---\n"
        f"REQUEST DATE (use this exact date for all tool calls): {request_date}\n\n"
        f"Customer request:\n{customer_request}\n\n"
        f"REMINDER: request date is {request_date}. Use it for every tool call."
    )
    result = orchestrator.run(full_task)
    return result

In [27]:
response = run_orchestrator(
    "I would like to request the following paper supplies for the ceremony: "
    "- 200 sheets of A4 glossy paper - 100 sheets of heavy cardstock (white) - "
    "100 sheets of colored paper (assorted colors) I need these supplies delivered "
    "by April 15, 2025. Thank you.",
    "2025-04-01",
)
print(response)

Dear Customer,

Thank you for your order of paper supplies for the ceremony. Here are the details of your order:
- 200 sheets of A4 glossy paper: Fulfilled ($9.50)
- 100 sheets of heavy cardstock (white): Fulfilled ($15.00)
- 100 sheets of colored paper (assorted colors): Fulfilled ($10.00)

Total Order Price: $34.50
Expected Delivery Date: April 15, 2025

Thank you for choosing Beaver's Choice Paper Company! We look forward to serving you again.


In [28]:
print(pb.generate_financial_report("2025-04-01")["cash_balance"])

45119.45


In [29]:
import time

def run_test_scenarios():
    print("Initializing Database...")
    pb.init_database(pb.db_engine)

    try:
        quote_requests_sample = pd.read_csv("quote_requests_sample.csv")
        quote_requests_sample["request_date"] = pd.to_datetime(
            quote_requests_sample["request_date"], format="%m/%d/%y", errors="coerce"
        )
        quote_requests_sample.dropna(subset=["request_date"], inplace=True)
        quote_requests_sample = quote_requests_sample.sort_values("request_date")
    except Exception as e:
        print(f"FATAL: Error loading test data: {e}")
        return

    initial_date = quote_requests_sample["request_date"].min().strftime("%Y-%m-%d")
    # Use check_finances (not pb.generate_financial_report) so inventory value covers
    # every item ever transacted, not just the original seeded 40% inventory snapshot.
    report = check_finances(initial_date)
    current_cash = report["cash_balance"]
    current_inventory = report["inventory_value"]

    results = []
    for idx, row in quote_requests_sample.iterrows():
        request_date = row["request_date"].strftime("%Y-%m-%d")

        print(f"\n=== Request {idx+1} ===")
        print(f"Context: {row['job']} organizing {row['event']}")
        print(f"Request Date: {request_date}")
        print(f"Cash Balance: ${current_cash:.2f}")
        print(f"Inventory Value: ${current_inventory:.2f}")

        request_with_date = f"{row['request']} (Date of request: {request_date})"

        try:
            response = run_orchestrator(request_with_date, request_date)
        except Exception as e:
            response = f"ERROR processing request: {e}"
            print(response)

        report = check_finances(request_date)
        current_cash = report["cash_balance"]
        current_inventory = report["inventory_value"]

        print(f"Response: {response}")
        print(f"Updated Cash: ${current_cash:.2f}")
        print(f"Updated Inventory: ${current_inventory:.2f}")

        results.append(
            {
                "request_id": idx + 1,
                "request_date": request_date,
                "cash_balance": current_cash,
                "inventory_value": current_inventory,
                "response": response,
            }
        )

        time.sleep(1)

    final_date = quote_requests_sample["request_date"].max().strftime("%Y-%m-%d")
    final_report = check_finances(final_date)
    print("\n===== FINAL FINANCIAL REPORT =====")
    print(f"Final Cash: ${final_report['cash_balance']:.2f}")
    print(f"Final Inventory: ${final_report['inventory_value']:.2f}")

    pd.DataFrame(results).to_csv("test_results.csv", index=False)
    return results


In [30]:
results = run_test_scenarios()

Initializing Database...

=== Request 1 ===
Context: office manager organizing ceremony
Request Date: 2025-04-01
Cash Balance: $45059.70
Inventory Value: $4940.30
Response: Dear Customer,

Thank you for your order. Here is the summary of your request:

1. **A4 glossy paper**: 200 sheets - **Fulfilled** - Total: $38.00
2. **Heavy cardstock (white)**: 100 sheets - **Fulfilled** - Total: $50.00
3. **Colored paper (assorted colors)**: 100 sheets - **Fulfilled** - Total: $10.00

Total Order Price: $98.00
Expected Delivery Date: April 15, 2025.

Thank you for choosing Beaver's Choice Paper Company.
Best regards,
Beaver's Choice Paper Company
Updated Cash: $45157.70
Updated Inventory: $4940.30

=== Request 2 ===
Context: hotel manager organizing parade
Request Date: 2025-04-03
Cash Balance: $45157.70
Inventory Value: $4940.30
Response: 
Dear Customer,

Thank you for your order request on 2025-04-03.

Unfortunately, we are unable to fulfill your order due to the following reasons:

1. Colorful

Reached max steps.

FUNC (get_supplier_delivery_date): Calculating for qty 5000 from date string '2025-04-04'
FUNC (get_supplier_delivery_date): Calculating for qty 5000 from date string '2025-04-04'
FUNC (get_supplier_delivery_date): Calculating for qty 5000 from date string '2025-04-04'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-04'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-08'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-12'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-16'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-20'


Reached max steps.

Code execution failed at line 'print(f"A4 Paper Status: {a4_status}")' due to: InterpreterError: The variable 
`a4_status` is not defined.

...

FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'


Code execution exceeded the maximum execution time of 30 seconds

FUNC (get_supplier_delivery_date): Calculating for qty 1000 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 2000 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 1000 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 1000 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'
FUNC (get_supplier_delivery_date): Calculating for qty 500 from date string '2025-04-14'
Response: 
Dear Customer,

Thank you for your order request placed on April 14, 2025, for supplies needed by April 15, 2025.

Unfortunately, we are unable to fulfill your entire order due to the 

Code execution exceeded the maximum execution time of 30 seconds

Response: 
Dear Customer,

Thank you for your order placed on April 14, 2025, for your upcoming ceremony. Here is the status of your requested paper supplies:

1. **High-quality white cardstock (500 sheets)**: Fulfilled - Total Price: $67.50
2. **Standard printing paper (1000 sheets)**: Rejected - Insufficient stock (order placed to replenish inventory).
3. **Colored paper (200 sheets)**: Fulfilled - Total Price: $19.00

**Total Order Price for fulfilled items**: $86.50.

Your order is expected to be delivered by April 15, 2025.

Thank you for choosing Beaver's Choice Paper Company!

Best regards,
Beaver's Choice Paper Company

Updated Cash: $46848.20
Updated Inventory: $4926.70

=== Request 19 ===
Context: city hall clerk organizing exhibition
Request Date: 2025-04-15
Cash Balance: $46848.20
Inventory Value: $4926.70
Response: {'fulfilled_items': {'Cardstock in assorted colors': {'status': 'Fulfilled', 'quantity': 1000, 'price_per_unit': 0.15, 'total_price': 150.0}}, 'rejected_items':

In [31]:
import pandas as pd
df = pd.read_csv("test_results.csv")
print(df.shape)
df[["request_id", "cash_balance", "inventory_value"]]

(20, 5)


,request_id,cash_balance,inventory_value
0,1,45157.7,4940.3
1,2,45157.7,4940.3
2,3,45157.7,4926.7
3,4,45225.2,4926.7
4,5,45400.7,4926.7
5,6,45685.7,4926.7
6,7,46293.2,4926.7
7,8,46293.2,4926.7
8,9,46293.2,4926.7
9,13,46388.2,4926.7


## Reflection

### Architecture recap
The diagram above splits the work along the same lines a human team would use:
someone who knows the warehouse (inventory_agent), someone who prices orders
(quoting_agent), and someone who is allowed to touch the books
(sales_agent) — with a single orchestrator (`CodeAgent`) deciding, per
customer request, which of those three to call and in what order. Keeping
`orchestrator` tool-less and routing everything through `managed_agents` was
a deliberate choice: it forces every database read/write to happen inside a
narrowly-scoped worker, so a bug in the orchestrator's plan can't itself
corrupt data — at worst it calls the wrong worker or skips a step, which is
visible in the final customer-facing response.

### Evaluation
`run_test_scenarios()` replays every row of `quote_requests_sample.csv` in
date order through `run_orchestrator`, and after each request re-reads
`check_finances()` so `test_results.csv` shows cash balance and inventory
value evolving request-by-request. Our 20-row run finished cleanly (final
cash $46,848.20, final inventory $4,926.70) but the printed step logs
surfaced four concrete, reproducible issues worth calling out:

- **`request_id` 9 hit a real type error mid-run.** The orchestrator's
  generated code executed `inventory_a4['found']` against a managed agent's
  reply and failed with `InterpreterError: ... TypeError: string indices
  must be integers, not 'str'`. Managed agents return a natural-language
  summary string, not the structured dict `check_inventory` itself returns —
  the orchestrator tried to index into prose as if it were the tool's raw
  return value. It recovered on the next step and still delivered a correct
  customer response (200 sheets A4 + 100 sheets A3 fulfilled, kraft
  envelopes rejected), but only because `CodeAgent` gets another attempt
  after an execution error — a stricter step budget would have surfaced this
  as a dropped request instead of a self-corrected one.
- **`request_id`s 3, 6, 8, and 14 each hit `"Reached max steps."`** In each
  case the step log shows `inventory_agent` calling
  `get_supplier_delivery_date` several times in a row with the *identical*
  quantity and date (e.g. qty 200 called five times for `request_id` 3
  before the agent moved on). The tool is deterministic, so repeat calls
  with the same arguments return the same answer — the agent wasn't
  gathering new information, just re-confirming its own prior tool call.
  This wasted 3–5 of the agent's 5-step budget per occurrence.
- **`request_id` 12 hit a `CodeAgent` parsing error.** The orchestrator's
  final answer arrived as plain prose instead of being wrapped in the
  `<code>...</code>` block `CodeAgent` requires to parse a step, producing
  `"Error in code parsing: ... regex pattern <code>(.*?)</code> was not
  found"`. It cost a retry step but did not change the final response.
- **`request_id` 3 also hit `"Code execution exceeded the maximum execution
  time of 30 seconds"`** during the same request that produced the repeated
  delivery-date calls above — consistent with the agent's step budget being
  consumed by redundant tool calls rather than genuine multi-step reasoning.

None of these four caused an incorrect final answer to reach the customer in
this run, but all four cost retried steps, and `request_id` 9 in particular
shows the orchestrator's generated code making an assumption about a managed
agent's return type that happens to be wrong.

### Improvements
1. **Return structured data from managed agents, not just prose.** The
   `request_id` 9 failure traces directly to `ToolCallingAgent`'s final
   answer being a summary string rather than the underlying tool's dict —
   the orchestrator has no reliable way to extract a field like `found`
   from it. Having each managed agent's final answer include a small
   embedded JSON block (or having the orchestrator call the underlying
   `@tool` functions directly for read-only lookups) would remove this
   class of `InterpreterError` entirely instead of relying on `CodeAgent`'s
   retry-after-failure behavior to paper over it.
2. **Detect and short-circuit repeated identical tool calls.** The
   `"Reached max steps"` cases all stem from an agent calling a
   deterministic tool (`get_supplier_delivery_date`) with arguments it has
   already used. A lightweight memoization wrapper around that tool (cache
   by `(as_of_date, quantity)` for the duration of one request) would let a
   repeat call return instantly with the cached result instead of costing a
   full step, freeing the step budget for requests that need it.
3. **A hard budget/inventory guardrail tool, not just a prompt instruction.**
   The "cash balance insufficient to justify a large reorder" rule currently
   lives only in `ORCHESTRATOR_INSTRUCTIONS` text. Moving that check into a
   tool (e.g. `can_afford_reorder(cost, as_of_date)`) that returns a hard
   True/False would make the constraint enforceable even if the orchestrator's
   plan drifts from the prompt on a longer request.
